# MySQL原始数据读取、Spark清洗、写入MySQL并上传HDFS

## 简介

本 notebook 按新的数据库流程运行：Spark 直接从 MySQL 原始分表读取当前批次数据，执行清洗转换，然后把同一份清洗结果写入 MySQL 清洗分表，并输出到 HDFS 的 cleaned 目录。

HDFS 上传不从清洗后的 MySQL 表二次读取，而是直接使用本次 Spark 清洗得到的 DataFrame。这样少一次数据库读取，也能保证 HDFS 文件和本次清洗结果完全一致。


## 1. 参数配置

主要修改 KEYWORD。清洗会读取该关键字当前批次数据，不再区分额外的数据范围。


In [1]:
from datetime import datetime
from pathlib import Path
import os
import re
import shutil
import subprocess
import sys
import tempfile

import pymysql

KEYWORD = '数据开发'  # 需要清洗的搜索关键词，必须和爬虫入库关键词一致
OUTPUT_FILENAME = ''  # HDFS 输出文件名；留空时自动生成“关键词_清洗.csv”

MYSQL_HOST = '127.0.0.1'  # MySQL 服务地址
MYSQL_PORT = 3306  # MySQL 服务端口
MYSQL_USER = 'root'  # MySQL 登录用户名
MYSQL_PASSWORD = os.environ.get('SHIXUN_MYSQL_PASSWORD', '1472580369@Lzh')  # MySQL 密码，优先读取环境变量
MYSQL_DATABASE = 'shixun'  # 项目使用的数据库名
RAW_JOB_TABLE = 'raw_job_info'  # 原始岗位信息表名
RAW_COMPANY_TABLE = 'raw_company_info'  # 原始公司信息表名
RAW_HR_TABLE = 'raw_hr_status_info'  # 原始 HR 状态表名
CLEAN_JOB_TABLE = 'clean_job_detail'  # 清洗后岗位明细表名
CLEAN_COMPANY_TABLE = 'clean_company_info'  # 清洗后公司信息表名
CLEAN_HR_TABLE = 'clean_hr_status_info'  # 清洗后 HR 状态表名
CLEAN_LOG_TABLE = 'clean_log'  # 清洗批次日志表名
KEEP_RECENT_BATCHES = 5  # 同一关键词最多保留的最近清洗批次数
MYSQL_JDBC_DRIVER = 'com.mysql.cj.jdbc.Driver'  # Spark JDBC 使用的 MySQL 驱动类名
MYSQL_JDBC_JAR = os.environ.get('MYSQL_JDBC_JAR', '')  # MySQL JDBC jar 路径；留空时自动从 Spark jars 查找

HDFS_URI = 'hdfs://localhost:9000'  # HDFS NameNode 地址
HDFS_USER = os.environ.get('HADOOP_USER_NAME') or os.environ.get('USERNAME') or 'hadoop'  # HDFS 用户名
HDFS_CLEAN_BASE_DIR = f'{HDFS_URI}/user/{HDFS_USER}/zhaopin/cleaned'  # HDFS 清洗结果根目录
SPARK_MASTER = 'local[2]'  # Spark 运行模式，local[2] 表示本机 2 个线程

# 元数据字段：贯穿 MySQL 清洗表和 HDFS 输出，用于标记批次、关键词和来源城市。
METADATA_COLUMNS = ['raw_job_id', 'run_id', 'batch_no', 'keyword', 'source_city', 'job_sign', 'is_current']
# 原始目标字段：从 MySQL 原始层读取后进入清洗逻辑的字段集合。
TARGET_COLUMNS = ['职位名称', '薪资', '薪资类型', '薪资发放次数', '工作城市', '行政区', '商圈/街道', '工作地点展示', '详细地址', '经度', '纬度', '经验要求', '学历要求', '工作类型', '工作模式', '职位类别', '公司名称', '公司规模', '公司性质', '融资阶段', '行业', '发布时间', '首次发布时间', '发布日期文本', '是否新职位', '招聘人数', 'HR状态', 'HR活跃标签', '职位标签汇总', '技能标签', '福利标签', '福利明细', '工作时间', '报告项/保障项', '职位描述', '职位亮点', '职位摘要', '认证/守护信息']
# 删除字段：清洗完成后不进入最终岗位明细表的中间字段。
DROP_COLUMNS = ['薪资类型', '薪资发放次数', '工作城市', '行政区', '商圈/街道', '详细地址', '经度', '纬度', '福利标签', '福利明细', '工作时间', '报告项/保障项', '首次发布时间', '发布日期文本', '是否新职位', '招聘人数', '职位亮点', '职位摘要', '职位标签汇总', '认证/守护信息']
# 最终业务字段：清洗后岗位明细保留的 19 个业务字段。
FINAL_COLUMNS = ['职位名称', '最低薪资', '最高薪资', '工作地点展示', '经验要求', '学历要求', '工作类型', '工作模式', '职位类别', '公司名称', '公司规模', '公司性质', '融资阶段', '行业', '发布时间', 'HR状态', 'HR活跃标签', '技能标签', '职位描述']
CLEAN_JOB_COLUMNS = METADATA_COLUMNS + FINAL_COLUMNS  # clean_job_detail 写入列顺序
# clean_company_info 写入列顺序。
CLEAN_COMPANY_COLUMNS = ['run_id', 'batch_no', 'keyword', 'source_city', 'company_sign', 'is_current', '公司名称', '公司规模', '公司性质', '融资阶段', '行业']
# clean_hr_status_info 写入列顺序。
CLEAN_HR_COLUMNS = ['run_id', 'batch_no', 'keyword', 'source_city', 'job_sign', 'hr_sign', 'is_current', '职位名称', '公司名称', 'HR状态', 'HR活跃标签']

DEFAULT_MONTHS = 12  # 薪资未标注几薪时默认按 12 薪计算
WORK_DAYS_PER_MONTH = 25  # 日薪换算月薪时使用的每月工作天数
HOURS_PER_DAY = 8  # 时薪换算日薪时使用的每日工作小时数
os.environ['PYSPARK_PYTHON'] = sys.executable  # Spark executor 使用当前 Python 解释器
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable  # Spark driver 使用当前 Python 解释器


def find_project_root(start_path=None):
    """
    Input:
    - start_path: 起始查找路径；为空时使用当前工作目录。
    Output:
    - Path，项目根目录。
    Function:
    - 定位当前项目根目录。
    """
    current = Path.cwd().resolve() if start_path is None else Path(start_path).resolve()
    while current != current.parent:
        if (current / '爬虫').is_dir() and (current / '数据清洗上传HDFS').is_dir():
            return current
        current = current.parent
    if (current / '爬虫').is_dir() and (current / '数据清洗上传HDFS').is_dir():
        return current
    raise FileNotFoundError('没有找到项目根目录：当前项目根目录应同时包含“爬虫”和“数据清洗上传HDFS”。')


def find_spark_home():
    """
    Input:
    - 无。
    Output:
    - Path，Spark 根目录。
    Function:
    - 查找 Spark 安装目录。
    """
    spark_home = os.environ.get('SPARK_HOME')
    if spark_home and (Path(spark_home) / 'python').exists():
        return Path(spark_home).resolve()
    spark_submit = shutil.which('spark-submit') or shutil.which('spark-submit.cmd')
    if spark_submit:
        return Path(spark_submit).resolve().parent.parent
    raise FileNotFoundError('没有找到 Spark，请配置 SPARK_HOME 或把 spark-submit 加入 PATH。')


def configure_pyspark_path(spark_home):
    """
    Input:
    - spark_home: Spark 安装目录。
    Output:
    - None。
    Function:
    - 配置 PySpark 模块搜索路径。
    """
    python_dir = spark_home / 'python'
    lib_dir = python_dir / 'lib'
    py4j_files = sorted(lib_dir.glob('py4j-*.zip'))
    for item in [python_dir, *py4j_files]:
        item_text = str(item)
        if item_text not in sys.path:
            sys.path.insert(0, item_text)


def find_mysql_jdbc_jar(spark_home, configured_jar=''):
    """
    Input:
    - spark_home: Spark 安装目录。
    - configured_jar: 手动指定的 MySQL JDBC jar 路径。
    Output:
    - str，JDBC jar 路径。
    Function:
    - 查找 MySQL JDBC 驱动 jar。
    """
    if configured_jar:
        jar_path = Path(configured_jar)
        if jar_path.exists():
            return str(jar_path.resolve())
        raise FileNotFoundError(f'MYSQL_JDBC_JAR 指定的文件不存在: {configured_jar}')
    candidates = sorted((spark_home / 'jars').glob('mysql-connector-j-*.jar'))
    if candidates:
        return str(candidates[0].resolve())
    raise FileNotFoundError('没有找到 mysql-connector-j-*.jar，请把 JDBC 驱动放到 Spark jars 目录，或设置 MYSQL_JDBC_JAR。')


def resolve_output_filename(output_filename, keyword):
    """
    Input:
    - output_filename: 手动指定的输出文件名；为空时自动生成。
    - keyword: 搜索关键词，也是后续批次筛选条件。
    Output:
    - str，CSV 文件名。
    Function:
    - 确定 HDFS 最终 CSV 文件名。
    """
    filename = output_filename or f'{keyword}_清洗.csv'
    return filename if filename.lower().endswith('.csv') else f'{filename}.csv'


PROJECT_ROOT = find_project_root()  # 项目根目录
SPARK_HOME = find_spark_home()  # Spark 安装目录
MYSQL_JDBC_JAR = find_mysql_jdbc_jar(SPARK_HOME, MYSQL_JDBC_JAR)  # MySQL JDBC 驱动路径
OUTPUT_FILE_NAME = resolve_output_filename(OUTPUT_FILENAME, KEYWORD)  # HDFS 最终输出文件名
configure_pyspark_path(SPARK_HOME)

from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import DoubleType, StringType

MYSQL_JDBC_URL = f'jdbc:mysql://{MYSQL_HOST}:{MYSQL_PORT}/{MYSQL_DATABASE}?useUnicode=true&characterEncoding=utf8&serverTimezone=Asia/Shanghai'  # Spark 连接 MySQL 的 JDBC URL

print('项目根目录:', PROJECT_ROOT)
print('MySQL原始表:', f'{MYSQL_DATABASE}.{RAW_JOB_TABLE}, {RAW_COMPANY_TABLE}, {RAW_HR_TABLE}')
print('MySQL清洗表:', f'{MYSQL_DATABASE}.{CLEAN_JOB_TABLE}, {CLEAN_COMPANY_TABLE}, {CLEAN_HR_TABLE}')
print('读取条件:', f'keyword={KEYWORD}, is_current=1')
print('HDFS清洗结果文件名:', OUTPUT_FILE_NAME)
print('Spark目录:', SPARK_HOME)
print('MySQL JDBC:', MYSQL_JDBC_JAR)
print('HDFS清洗结果根目录:', HDFS_CLEAN_BASE_DIR)


项目根目录: D:\桌面\实训项目
MySQL原始表: shixun.raw_job_info, raw_company_info, raw_hr_status_info
MySQL清洗表: shixun.clean_job_detail, clean_company_info, clean_hr_status_info
读取条件: keyword=数据开发, is_current=1
HDFS清洗结果文件名: 数据开发_清洗.csv
Spark目录: D:\Spark\spark-3.5.1-bin-hadoop3
MySQL JDBC: D:\Spark\spark-3.5.1-bin-hadoop3\jars\mysql-connector-j-8.0.33.jar
HDFS清洗结果根目录: hdfs://localhost:9000/user/10967/zhaopin/cleaned


## 2. 创建 SparkSession

Spark 负责从 MySQL 原始表读取数据、执行清洗转换，并把清洗结果写入 MySQL 和 HDFS。


In [2]:
def create_spark_session(app_name):
    """
    Input:
    - app_name: Spark 应用名称。
    Output:
    - SparkSession。
    Function:
    - 创建 SparkSession。
    """
    active_session = SparkSession.getActiveSession()
    if active_session is not None:
        active_session.stop()

    builder = (
        SparkSession.builder
        .appName(app_name)
        .master(SPARK_MASTER)
        .config('spark.hadoop.fs.defaultFS', HDFS_URI)
        .config('spark.pyspark.python', sys.executable)
        .config('spark.pyspark.driver.python', sys.executable)
        .config('spark.sql.shuffle.partitions', '2')
    )
    if MYSQL_JDBC_JAR:
        builder = (
            builder
            .config('spark.jars', MYSQL_JDBC_JAR)
            .config('spark.driver.extraClassPath', MYSQL_JDBC_JAR)
            .config('spark.executor.extraClassPath', MYSQL_JDBC_JAR)
        )
    return builder.getOrCreate()


spark = create_spark_session('ZhaopinSparkCleaning')
print('Spark版本:', spark.version)


Spark版本: 3.5.1


## 3. 从MySQL原始表读取

直接从原始岗位、公司、HR 表读取当前 keyword 的当前批次数据，不再依赖本地 CSV，也不再把原始 CSV 上传 HDFS。


In [3]:
def find_hdfs_command():
    """
    Input:
    - 无。
    Output:
    - str，hdfs 命令路径。
    Function:
    - 查找 hdfs 命令。
    """
    hdfs_command = shutil.which('hdfs') or shutil.which('hdfs.cmd')
    if hdfs_command is None:
        raise FileNotFoundError('没有找到 hdfs 命令，请先配置 Hadoop bin 目录到 PATH。')
    return hdfs_command


HDFS_COMMAND = find_hdfs_command()


def run_hdfs_dfs(args, show_output=False):
    """
    Input:
    - args: 传给 hdfs dfs 的命令参数。
    - show_output: 是否打印命令标准输出。
    Output:
    - CompletedProcess，命令执行结果。
    Function:
    - 执行 hdfs dfs 命令。
    """
    completed = subprocess.run(
        [HDFS_COMMAND, 'dfs', *[str(item) for item in args]],
        check=True,
        capture_output=True,
        text=True,
        encoding='utf-8',
        errors='replace',
    )
    if show_output and completed.stdout.strip():
        print(completed.stdout.strip())
    if completed.stderr.strip():
        print(completed.stderr.strip())
    return completed


def qi(name):
    """
    Input:
    - name: 字段名。
    Output:
    - str，转义后的字段名。
    Function:
    - 给 MySQL 字段名加反引号。
    """
    return f'{chr(96)}{name}{chr(96)}'


def mysql_literal(value):
    """
    Input:
    - value: 需要清洗、格式化或转义的输入值。
    Output:
    - str，SQL 字面量。
    Function:
    - 把值转为 MySQL 字符串字面量。
    """
    text = '' if value is None else str(value)
    return "'" + text.replace('\\', '\\\\').replace("'", "''") + "'"


def mysql_jdbc_options():
    """
    Input:
    - 无。
    Output:
    - dict，JDBC 配置。
    Function:
    - 汇总 Spark JDBC 连接参数。
    """
    return {'url': MYSQL_JDBC_URL, 'driver': MYSQL_JDBC_DRIVER, 'user': MYSQL_USER, 'password': MYSQL_PASSWORD}


def build_raw_mysql_query(keyword):
    """
    Input:
    - keyword: 搜索关键词，也是后续批次筛选条件。
    Output:
    - str，JDBC dbtable 子查询。
    Function:
    - 构造 Spark JDBC 读取原始层的 SQL 子查询。
    """
    def j_col(name):
        """
        Input:
        - name: 字段名。
        Output:
        - str，SQL 字段片段。
        Function:
        - 生成岗位表字段选择 SQL。
        """
        return f'j.{qi(name)} AS {qi(name)}'
    def c_col(name):
        """
        Input:
        - name: 字段名。
        Output:
        - str，SQL 字段片段。
        Function:
        - 生成公司表字段选择 SQL。
        """
        return f'COALESCE(c.{qi(name)}, {mysql_literal("")}) AS {qi(name)}'
    def h_col(name):
        """
        Input:
        - name: 字段名。
        Output:
        - str，SQL 字段片段。
        Function:
        - 生成 HR 表字段选择 SQL。
        """
        return f'COALESCE(h.{qi(name)}, {mysql_literal("")}) AS {qi(name)}'

    raw_columns = [
        'j.id AS raw_job_id', 'j.run_id AS run_id', 'j.batch_no AS batch_no', 'j.keyword AS keyword',
        'j.source_city AS source_city', 'j.job_sign AS job_sign', 'j.is_current AS is_current',
        j_col('职位名称'), j_col('薪资'), j_col('薪资类型'), j_col('薪资发放次数'), j_col('工作城市'), j_col('行政区'),
        j_col('商圈/街道'), j_col('工作地点展示'), j_col('详细地址'), j_col('经度'), j_col('纬度'), j_col('经验要求'),
        j_col('学历要求'), j_col('工作类型'), j_col('工作模式'), j_col('职位类别'), j_col('公司名称'), c_col('公司规模'),
        c_col('公司性质'), c_col('融资阶段'), c_col('行业'), j_col('发布时间'), j_col('首次发布时间'), j_col('发布日期文本'),
        j_col('是否新职位'), j_col('招聘人数'), h_col('HR状态'), h_col('HR活跃标签'), j_col('职位标签汇总'), j_col('技能标签'),
        j_col('福利标签'), j_col('福利明细'), j_col('工作时间'), j_col('报告项/保障项'), j_col('职位描述'), j_col('职位亮点'),
        j_col('职位摘要'), j_col('认证/守护信息')
    ]
    where_clause = f'j.keyword = {mysql_literal(keyword)} AND j.is_current = 1'
    company_join = (
        f'LEFT JOIN {RAW_COMPANY_TABLE} c ON c.run_id = j.run_id '
        f'AND c.keyword = j.keyword '
        f'AND c.source_city <=> j.source_city AND c.{qi("公司名称")} <=> j.{qi("公司名称")}'
    )
    hr_join = f'LEFT JOIN {RAW_HR_TABLE} h ON h.run_id = j.run_id AND h.job_sign <=> j.job_sign'
    return f'(SELECT {", ".join(raw_columns)} FROM {RAW_JOB_TABLE} j {company_join} {hr_join} WHERE {where_clause}) AS raw_joined'


def read_source_data(spark, keyword=KEYWORD):
    """
    Input:
    - spark: SparkSession 对象。
    - keyword: 搜索关键词，也是后续批次筛选条件。
    Output:
    - DataFrame，MySQL 联表数据。
    Function:
    - 从 MySQL 读取当前原始批次数据。
    """
    options = mysql_jdbc_options()
    return (
        spark.read.format('jdbc')
        .option('url', options['url'])
        .option('driver', options['driver'])
        .option('dbtable', build_raw_mysql_query(keyword))
        .option('user', options['user'])
        .option('password', options['password'])
        .load()
    )


def select_target_columns(df):
    """
    Input:
    - df: 待处理的 DataFrame。
    Output:
    - DataFrame，目标字段数据。
    Function:
    - 选择清洗所需字段并统一空值。
    """
    selected_columns = []
    for column in METADATA_COLUMNS:
        if column in {'raw_job_id', 'batch_no', 'is_current'}:
            expression = F.col(column).cast('long') if column in df.columns else F.lit(None).cast('long')
        else:
            expression = F.trim(F.coalesce(F.col(column).cast('string'), F.lit(''))) if column in df.columns else F.lit('')
        selected_columns.append(expression.alias(column))

    for column in TARGET_COLUMNS:
        if column in df.columns:
            selected_columns.append(F.trim(F.coalesce(F.col(column).cast('string'), F.lit(''))).alias(column))
        else:
            selected_columns.append(F.lit('').alias(column))
    return df.select(*selected_columns)


def map_stage(spark, keyword=KEYWORD):
    """
    Input:
    - spark: SparkSession 对象。
    - keyword: 搜索关键词，也是后续批次筛选条件。
    Output:
    - DataFrame，Map 输出。
    Function:
    - 执行清洗 Map 阶段。
    """
    print('读取MySQL原始表:', f'{MYSQL_DATABASE}.{RAW_JOB_TABLE}, {RAW_COMPANY_TABLE}, {RAW_HR_TABLE}')
    print('读取条件:', f'keyword={keyword}, is_current=1')
    source_df = read_source_data(spark, keyword)
    mapped_df = select_target_columns(source_df)
    print('Map 输出记录数:', mapped_df.count())
    return mapped_df


## 4. Spark 清洗数据

统一薪资维度并拆分最低/最高薪资、清洗发布时间，并删除不需要字段。

In [4]:
# 作用：格式化万元数值，去掉小数末尾多余的 0。
# 输入：value，数值类型的万元金额。
# 输出：字符串，适合写入薪资拆分字段的万元文本。
def format_wan(value):
    """
    Input:
    - value: 需要清洗、格式化或转义的输入值。
    Output:
    - str，万元文本。
    Function:
    - 格式化万元数值。
    """
    text = f'{float(value):.2f}'
    return text.rstrip('0').rstrip('.')


# 作用：把 Spark 数值列格式化为万元文本数字。
# 输入：column，Spark 数值列。
# 输出：Column，去掉多余 0 的字符串列。
def format_wan_column(column):
    """
    Input:
    - column: Spark 数值列。
    Output:
    - Column，万元文本列。
    Function:
    - 格式化 Spark 万元数值列。
    """
    formatted = F.format_string('%.2f', column)
    formatted = F.regexp_replace(formatted, '0+$', '')
    return F.regexp_replace(formatted, '\\.$', '')


# 作用：从薪资文本中提取发薪月份，未标注时默认 12 薪。
# 输入：salary_col，Spark 薪资文本列。
# 输出：Column，发薪月份数值列。
def extract_salary_months(salary_col):
    """
    Input:
    - salary_col: 原始薪资文本列。
    Output:
    - Column，发薪月份列。
    Function:
    - 从薪资文本提取发薪月份。
    """
    month_text = F.regexp_extract(salary_col, r'(\d+(?:\.\d+)?)\s*薪', 1)
    return F.when(month_text != '', month_text.cast('double')).otherwise(F.lit(float(DEFAULT_MONTHS)))


# 作用：从薪资主体中提取薪资下限和上限。
# 输入：salary_body_col，去掉“13薪”等后缀后的薪资主体列。
# 输出：二元组 (low_col, high_col)，都是 Spark 数值列。
def extract_salary_bounds(salary_body_col):
    """
    Input:
    - salary_body_col: 去掉发薪月份后缀的薪资主体列。
    Output:
    - tuple，下限列和上限列。
    Function:
    - 提取薪资上下限。
    """
    first_number = F.regexp_extract(salary_body_col, r'(\d+(?:\.\d+)?)', 1)
    second_number = F.regexp_extract(salary_body_col, r'\d+(?:\.\d+)?\D+(\d+(?:\.\d+)?)', 1)

    first_value = F.when(first_number != '', first_number.cast('double'))
    second_value = F.when(second_number != '', second_number.cast('double')).otherwise(first_value)
    low_col = F.least(first_value, second_value)
    high_col = F.greatest(first_value, second_value)
    return low_col, high_col


# 作用：按薪资单位把薪资上下限换算为万元/年。
# 输入：salary_body_col 薪资主体列，low_col 下限列，high_col 上限列，months_col 发薪月份列。
# 输出：二元组 (annual_low_col, annual_high_col)，单位都是万元/年。
def convert_salary_bounds_to_annual(salary_body_col, low_col, high_col, months_col):
    """
    Input:
    - salary_body_col: 去掉发薪月份后缀的薪资主体列。
    - low_col: 薪资下限列。
    - high_col: 薪资上限列。
    - months_col: 发薪月份列。
    Output:
    - tuple，年薪下限列和上限列。
    Function:
    - 把不同单位薪资换算为万元每年。
    """
    is_day_salary = salary_body_col.contains('元/天') | salary_body_col.contains('/天')
    is_hour_salary = (
        salary_body_col.contains('元/时') |
        salary_body_col.contains('/时') |
        salary_body_col.contains('元/小时') |
        salary_body_col.contains('/小时')
    )
    is_k_salary = salary_body_col.rlike('[kK]')
    is_wan_salary = salary_body_col.contains('万')
    is_qian_salary = salary_body_col.contains('千')
    is_yuan_salary = salary_body_col.contains('元') | (F.greatest(low_col, high_col) >= F.lit(1000.0))

    day_factor = F.lit(float(WORK_DAYS_PER_MONTH * 12 / 10000))
    hour_factor = F.lit(float(HOURS_PER_DAY * WORK_DAYS_PER_MONTH * 12 / 10000))

    annual_low = (
        F.when(low_col.isNull(), F.lit(None).cast('double'))
        .when(is_day_salary, low_col * day_factor)
        .when(is_hour_salary, low_col * hour_factor)
        .when(is_k_salary, low_col * months_col / F.lit(10.0))
        .when(is_wan_salary, low_col * months_col)
        .when(is_qian_salary, low_col * F.lit(0.1) * months_col)
        .when(is_yuan_salary, low_col * months_col / F.lit(10000.0))
        .otherwise(low_col * months_col)
    )
    annual_high = (
        F.when(high_col.isNull(), F.lit(None).cast('double'))
        .when(is_day_salary, high_col * day_factor)
        .when(is_hour_salary, high_col * hour_factor)
        .when(is_k_salary, high_col * months_col / F.lit(10.0))
        .when(is_wan_salary, high_col * months_col)
        .when(is_qian_salary, high_col * F.lit(0.1) * months_col)
        .when(is_yuan_salary, high_col * months_col / F.lit(10000.0))
        .otherwise(high_col * months_col)
    )
    return annual_low, annual_high


# 作用：把年薪数值列格式化为最终薪资拆分字段。
# 输入：annual_col 年薪列，fill_salary 无法解析时的回填数值。
# 输出：Column，写入“最低薪资/最高薪资”的万元/年文本。
def build_salary_value_column(annual_col, fill_salary):
    """
    Input:
    - annual_col: 换算后的年薪数值列。
    - fill_salary: 薪资无法解析时使用的平均值回填。
    Output:
    - Column，薪资文本列。
    Function:
    - 生成最终薪资字段文本。
    """
    fill_text = f'{format_wan(fill_salary)}万' if fill_salary is not None else ''
    return F.when(
        annual_col.isNull(),
        F.lit(fill_text),
    ).otherwise(F.concat(format_wan_column(annual_col), F.lit('万')))


# 作用：把发布时间统一清洗为 YYYY-MM-DD 格式。
# 输入：date_col，原始发布时间列。
# 输出：Column，标准日期文本列。
def clean_publish_date_column(date_col):
    """
    Input:
    - date_col: 原始发布时间列。
    Output:
    - Column，日期文本列。
    Function:
    - 统一发布时间格式。
    """
    raw = F.trim(F.coalesce(date_col.cast('string'), F.lit('')))
    year = F.regexp_extract(raw, r'(\d{4})[-/.年](\d{1,2})[-/.月](\d{1,2})', 1)
    month = F.regexp_extract(raw, r'(\d{4})[-/.年](\d{1,2})[-/.月](\d{1,2})', 2)
    day = F.regexp_extract(raw, r'(\d{4})[-/.年](\d{1,2})[-/.月](\d{1,2})', 3)
    return F.when(
        year != '',
        F.concat(year, F.lit('-'), F.lpad(month, 2, '0'), F.lit('-'), F.lpad(day, 2, '0')),
    ).otherwise(raw)


# 作用：用 Spark 内置表达式完成薪资、日期和字段删除清洗。
# 输入：mapped_df，Map 阶段输出的 Spark DataFrame。
# 输出：DataFrame，最终可写入 HDFS 的清洗结果。
def reduce_stage(mapped_df):
    """
    Input:
    - mapped_df: Map 阶段输出的 Spark DataFrame。
    Output:
    - DataFrame，最终清洗结果。
    Function:
    - 执行清洗 Reduce 阶段。
    """
    raw_salary = F.trim(F.coalesce(F.col('薪资').cast('string'), F.lit('')))
    normalized_salary = F.regexp_replace(raw_salary, '[－—–~～至]', '-')
    salary_body = F.regexp_replace(normalized_salary, r'[·・\s]*\d+(?:\.\d+)?\s*薪.*$', '')
    months_col = extract_salary_months(normalized_salary)
    low_col, high_col = extract_salary_bounds(salary_body)
    annual_low_col, annual_high_col = convert_salary_bounds_to_annual(salary_body, low_col, high_col, months_col)

    df = (
        mapped_df
        .withColumn('_annual_low', annual_low_col)
        .withColumn('_annual_high', annual_high_col)
        .withColumn('_salary_average', (F.col('_annual_low') + F.col('_annual_high')) / F.lit(2.0))
    )

    average_row = df.select(F.avg('_salary_average').alias('average_salary')).first()
    average_salary = average_row['average_salary'] if average_row else None

    df = (
        df
        .withColumn('最低薪资', build_salary_value_column(F.col('_annual_low'), average_salary))
        .withColumn('最高薪资', build_salary_value_column(F.col('_annual_high'), average_salary))
        .drop('薪资', '_annual_low', '_annual_high', '_salary_average')
    )

    if '发布时间' in df.columns:
        df = df.withColumn('发布时间', clean_publish_date_column(F.col('发布时间')))

    existing_drop_columns = [column for column in DROP_COLUMNS if column in df.columns]
    df = df.drop(*existing_drop_columns)

    for column in METADATA_COLUMNS:
        if column not in df.columns:
            if column in {'raw_job_id', 'batch_no', 'is_current'}:
                df = df.withColumn(column, F.lit(None).cast('long'))
            else:
                df = df.withColumn(column, F.lit(''))

    for column in FINAL_COLUMNS:
        if column not in df.columns:
            df = df.withColumn(column, F.lit(''))
    return df.select(*CLEAN_JOB_COLUMNS)


## 5. 清洗结果写入MySQL和HDFS

清洗结果先写入 MySQL 三张清洗表，再把同一份 Spark DataFrame 写入 HDFS。只有写入 HDFS 成功后，才把该批次标记为当前批次并追加清洗日志。


In [5]:
def get_mysql_connection():
    """
    Input:
    - 无。
    Output:
    - Connection，MySQL 连接。
    Function:
    - 创建 pymysql 连接。
    """
    return pymysql.connect(
        host=MYSQL_HOST,
        port=int(MYSQL_PORT),
        user=MYSQL_USER,
        password=MYSQL_PASSWORD,
        database=MYSQL_DATABASE,
        charset='utf8mb4',
        autocommit=False,
    )


def delete_existing_clean_run(run_id):
    """
    Input:
    - run_id: 批次唯一编号。
    Output:
    - None。
    Function:
    - 删除同 run_id 已存在的清洗数据。
    """
    connection = get_mysql_connection()
    try:
        with connection.cursor() as cursor:
            for table in [CLEAN_JOB_TABLE, CLEAN_COMPANY_TABLE, CLEAN_HR_TABLE]:
                cursor.execute(f'DELETE FROM {table} WHERE run_id = %s', (run_id,))
            cursor.execute(f'DELETE FROM {CLEAN_LOG_TABLE} WHERE run_id = %s', (run_id,))
        connection.commit()
    except Exception:
        connection.rollback()
        raise
    finally:
        connection.close()


def write_table_jdbc(df, table_name):
    """
    Input:
    - df: 待处理的 DataFrame。
    - table_name: 目标 MySQL 表名。
    Output:
    - None。
    Function:
    - 通过 Spark JDBC 写入 MySQL 表。
    """
    options = mysql_jdbc_options()
    (
        df.write.format('jdbc')
        .option('url', options['url'])
        .option('driver', options['driver'])
        .option('dbtable', table_name)
        .option('user', options['user'])
        .option('password', options['password'])
        .mode('append')
        .save()
    )


def build_clean_company_df(cleaned_df):
    """
    Input:
    - cleaned_df: 清洗后的岗位明细 DataFrame。
    Output:
    - DataFrame，公司清洗数据。
    Function:
    - 从岗位清洗结果生成公司清洗表。
    """
    return (
        cleaned_df
        .withColumn(
            'company_sign',
            F.md5(F.concat_ws('|', F.col('keyword'), F.coalesce(F.col('source_city'), F.lit('')), F.coalesce(F.col('公司名称'), F.lit('')))),
        )
        .select(*CLEAN_COMPANY_COLUMNS)
        .dropDuplicates(['run_id', 'keyword', 'source_city', '公司名称'])
    )


def build_clean_hr_df(cleaned_df):
    """
    Input:
    - cleaned_df: 清洗后的岗位明细 DataFrame。
    Output:
    - DataFrame，HR 清洗数据。
    Function:
    - 从岗位清洗结果生成 HR 清洗表。
    """
    return (
        cleaned_df
        .withColumn(
            'hr_sign',
            F.md5(F.concat_ws('|', F.col('job_sign'), F.coalesce(F.col('HR状态'), F.lit('')), F.coalesce(F.col('HR活跃标签'), F.lit('')))),
        )
        .select(*CLEAN_HR_COLUMNS)
        .dropDuplicates(['run_id', 'job_sign', 'hr_sign'])
    )


def write_clean_tables_to_mysql(cleaned_df):
    """
    Input:
    - cleaned_df: 清洗后的岗位明细 DataFrame。
    Output:
    - dict，写入统计。
    Function:
    - 写入三张 MySQL 清洗表。
    """
    first_row = cleaned_df.select('run_id', 'batch_no', 'keyword').first()
    if first_row is None:
        raise ValueError('清洗结果为空，无法写入 MySQL。')

    run_id = first_row['run_id']
    batch_no = int(first_row['batch_no'])
    keyword = first_row['keyword']

    delete_existing_clean_run(run_id)

    db_job_df = cleaned_df.select(*CLEAN_JOB_COLUMNS).withColumn('is_current', F.lit(0))
    db_company_df = build_clean_company_df(cleaned_df).withColumn('is_current', F.lit(0))
    db_hr_df = build_clean_hr_df(cleaned_df).withColumn('is_current', F.lit(0))

    write_table_jdbc(db_job_df, CLEAN_JOB_TABLE)
    write_table_jdbc(db_company_df, CLEAN_COMPANY_TABLE)
    write_table_jdbc(db_hr_df, CLEAN_HR_TABLE)

    return {
        'run_id': run_id,
        'batch_no': batch_no,
        'keyword': keyword,
        'job_count': db_job_df.count(),
        'company_count': db_company_df.count(),
        'hr_count': db_hr_df.count(),
    }


def enforce_clean_retention(cursor, keyword):
    """
    Input:
    - cursor: pymysql 游标对象。
    - keyword: 搜索关键词，也是后续批次筛选条件。
    Output:
    - list[str]，被删除的 run_id 列表。
    Function:
    - 执行清洗层最近批次保留策略。
    """
    cursor.execute(
        f'''
        SELECT run_id
        FROM {CLEAN_LOG_TABLE}
        WHERE keyword = %s AND is_retained = 1
        ORDER BY batch_no DESC, log_id DESC
        ''',
        (keyword,),
    )
    old_run_ids = [row[0] for row in cursor.fetchall()[KEEP_RECENT_BATCHES:]]
    for old_run_id in old_run_ids:
        for table in [CLEAN_JOB_TABLE, CLEAN_COMPANY_TABLE, CLEAN_HR_TABLE]:
            cursor.execute(f'DELETE FROM {table} WHERE run_id = %s', (old_run_id,))
        cursor.execute(
            f'UPDATE {CLEAN_LOG_TABLE} SET is_retained = 0, is_current = 0, deleted_at = NOW() WHERE run_id = %s',
            (old_run_id,),
        )
    return old_run_ids


def finalize_clean_success(mysql_result, hdfs_output_file, source_cities, start_time, end_time):
    """
    Input:
    - mysql_result: MySQL 写入结果统计。
    - hdfs_output_file: HDFS 最终输出文件路径。
    - source_cities: 本批次包含的城市列表。
    - start_time: 任务开始时间。
    - end_time: 任务结束时间。
    Output:
    - list[str]，被删除的 run_id 列表。
    Function:
    - 更新清洗成功状态并写入日志。
    """
    connection = get_mysql_connection()
    try:
        with connection.cursor() as cursor:
            keyword = mysql_result['keyword']
            run_id = mysql_result['run_id']

            for table in [CLEAN_JOB_TABLE, CLEAN_COMPANY_TABLE, CLEAN_HR_TABLE]:
                cursor.execute(f'UPDATE {table} SET is_current = 0 WHERE keyword = %s', (keyword,))
                cursor.execute(f'UPDATE {table} SET is_current = 1 WHERE run_id = %s', (run_id,))

            cursor.execute(f'UPDATE {CLEAN_LOG_TABLE} SET is_current = 0 WHERE keyword = %s', (keyword,))
            cursor.execute(f'DELETE FROM {CLEAN_LOG_TABLE} WHERE run_id = %s', (run_id,))
            cursor.execute(
                f'''
                INSERT INTO {CLEAN_LOG_TABLE}
                (run_id, batch_no, keyword, source_cities, job_record_count, company_record_count,
                 hr_record_count, hdfs_output_path, run_status, run_message, start_time, end_time,
                 is_current, is_retained, created_at)
                VALUES (%s, %s, %s, %s, %s, %s, %s, %s, 'success', '清洗入库和HDFS上传完成', %s, %s, 1, 1, %s)
                ''',
                (
                    run_id,
                    mysql_result['batch_no'],
                    keyword,
                    ','.join(source_cities),
                    mysql_result['job_count'],
                    mysql_result['company_count'],
                    mysql_result['hr_count'],
                    hdfs_output_file,
                    start_time.strftime('%Y-%m-%d %H:%M:%S'),
                    end_time.strftime('%Y-%m-%d %H:%M:%S'),
                    end_time.strftime('%Y-%m-%d %H:%M:%S'),
                ),
            )
            deleted_run_ids = enforce_clean_retention(cursor, keyword)
        connection.commit()
        return deleted_run_ids
    except Exception:
        connection.rollback()
        raise
    finally:
        connection.close()


def get_hdfs_filesystem(spark):
    """
    Input:
    - spark: SparkSession 对象。
    Output:
    - tuple，FileSystem 和 Path 类。
    Function:
    - 获取 Hadoop FileSystem 和 Path 类。
    """
    hadoop_conf = spark._jsc.hadoopConfiguration()
    fs = spark._jvm.org.apache.hadoop.fs.FileSystem.get(hadoop_conf)
    path_class = spark._jvm.org.apache.hadoop.fs.Path
    return fs, path_class


def find_part_csv_file(fs, PathClass, temp_output_dir):
    """
    Input:
    - fs: Hadoop FileSystem 对象。
    - PathClass: Hadoop Path 类。
    - temp_output_dir: Spark CSV 临时输出目录。
    Output:
    - Path，part CSV 路径。
    Function:
    - 查找 Spark 输出的 part CSV 文件。
    """
    statuses = fs.listStatus(PathClass(temp_output_dir))
    for status in statuses:
        name = status.getPath().getName()
        if name.startswith('part-') and name.endswith('.csv'):
            return status.getPath()
    raise FileNotFoundError(f'没有在 Spark 输出目录中找到 part-*.csv: {temp_output_dir}')


def copy_part_file_to_utf8_sig(part_file, output_file):
    """
    Input:
    - part_file: Spark 生成的 part CSV 文件路径。
    - output_file: HDFS 最终 CSV 文件路径。
    Output:
    - None。
    Function:
    - 给 part CSV 加 UTF-8 BOM 并写回 HDFS。
    """
    hdfs_command = globals().get('HDFS_COMMAND') or find_hdfs_command()
    with tempfile.TemporaryDirectory(prefix='zhaopin_hdfs_utf8sig_') as temp_dir:
        temp_dir = Path(temp_dir)
        local_part_file = temp_dir / 'part.csv'
        local_output_file = temp_dir / 'final_utf8_sig.csv'
        subprocess.run([hdfs_command, 'dfs', '-copyToLocal', '-f', str(part_file), str(local_part_file)], check=True)
        with local_part_file.open('rb') as source_file, local_output_file.open('wb') as target_file:
            target_file.write(b'\xef\xbb\xbf')
            shutil.copyfileobj(source_file, target_file)
        subprocess.run([hdfs_command, 'dfs', '-put', '-f', str(local_output_file), output_file], check=True)


def write_to_hdfs(cleaned_df, keyword, output_filename):
    """
    Input:
    - cleaned_df: 清洗后的岗位明细 DataFrame。
    - keyword: 搜索关键词，也是后续批次筛选条件。
    - output_filename: 手动指定的输出文件名；为空时自动生成。
    Output:
    - str，HDFS 输出文件路径。
    Function:
    - 把清洗结果写入 HDFS 单个 CSV 文件。
    """
    hdfs_output_dir = f'{HDFS_CLEAN_BASE_DIR}/{keyword}'
    hdfs_temp_dir = f'{hdfs_output_dir}/_tmp_{output_filename.replace(".csv", "")}'
    hdfs_output_file = f'{hdfs_output_dir}/{output_filename}'

    fs, PathClass = get_hdfs_filesystem(spark)
    legacy_output_dir = hdfs_output_file[:-4] if hdfs_output_file.endswith('.csv') else hdfs_output_file
    fs.delete(PathClass(hdfs_temp_dir), True)
    fs.delete(PathClass(hdfs_output_file), False)
    fs.delete(PathClass(legacy_output_dir), True)
    fs.mkdirs(PathClass(hdfs_output_dir))

    (
        cleaned_df.coalesce(1)
        .write
        .mode('overwrite')
        .option('header', True)
        .option('encoding', 'UTF-8')
        .csv(hdfs_temp_dir)
    )

    part_file = find_part_csv_file(fs, PathClass, hdfs_temp_dir)
    copy_part_file_to_utf8_sig(part_file, hdfs_output_file)
    fs.delete(PathClass(hdfs_temp_dir), True)
    return hdfs_output_file


## 6. 总调用

运行下面单元格即可完成：MySQL 原始分表读取 -> Spark 清洗 -> 清洗结果写入 MySQL -> 同一份清洗结果上传 HDFS -> 记录清洗日志。


In [6]:
def run_spark_cleaning(keyword=KEYWORD):
    """
    Input:
    - keyword: 搜索关键词，也是后续批次筛选条件。
    Output:
    - tuple，HDFS 路径和清洗 DataFrame。
    Function:
    - 串联读取、清洗、写库、写 HDFS 和日志更新。
    """
    clean_start_time = datetime.now()
    mapped_df = map_stage(spark, keyword)
    cleaned_df = reduce_stage(mapped_df).cache()
    record_count = cleaned_df.count()
    if record_count == 0:
        raise ValueError(f'MySQL原始表没有可清洗数据: keyword={keyword}, is_current=1')

    source_cities = [row['source_city'] for row in cleaned_df.select('source_city').distinct().collect() if row['source_city']]
    mysql_result = write_clean_tables_to_mysql(cleaned_df)
    hdfs_output_file = write_to_hdfs(cleaned_df, keyword, OUTPUT_FILE_NAME)
    clean_end_time = datetime.now()
    deleted_run_ids = finalize_clean_success(mysql_result, hdfs_output_file, source_cities, clean_start_time, clean_end_time)

    print('清洗完成')
    print('记录数:', record_count)
    print('字段数:', len(cleaned_df.columns))
    print('批次号 batch_no:', mysql_result['batch_no'])
    print('批次ID run_id:', mysql_result['run_id'])
    print('MySQL清洗岗位记录数:', mysql_result['job_count'])
    print('MySQL清洗公司记录数:', mysql_result['company_count'])
    print('MySQL清洗HR记录数:', mysql_result['hr_count'])
    print('HDFS 清洗输出文件:', hdfs_output_file)
    if deleted_run_ids:
        print('超过最近5批，已删除最老清洗业务批次:', ', '.join(deleted_run_ids))
    print('说明: MySQL 清洗业务表同 keyword 保留最近5批；HDFS 同名清洗结果文件覆盖写入。')
    return hdfs_output_file, cleaned_df


HDFS_OUTPUT_FILE, result_df = run_spark_cleaning(KEYWORD)
result_df.show(5, truncate=False)


读取MySQL原始表: shixun.raw_job_info, raw_company_info, raw_hr_status_info
读取条件: keyword=数据开发, is_current=1
Map 输出记录数: 1822
清洗完成
记录数: 1822
字段数: 26
批次号 batch_no: 1
批次ID run_id: 20260618_082950_数据开发_batch1
MySQL清洗岗位记录数: 1822
MySQL清洗公司记录数: 1276
MySQL清洗HR记录数: 1822
HDFS 清洗输出文件: hdfs://localhost:9000/user/10967/zhaopin/cleaned/数据开发/数据开发_清洗.csv
说明: MySQL 清洗业务表同 keyword 保留最近5批；HDFS 同名清洗结果文件覆盖写入。
+----------+-------------------------------+--------+--------+-----------+--------------------------------+----------+--------------------------------------------+--------+--------+----------------+--------+--------+--------+--------+------------+----------------------------------+-----------+--------+---------+-----------+----------+--------------+-----------------------+--------------------------------------------------------------------------------------------------------------+--------------------------------------------------------------------------------------------------------------------------------